# Pf display-feature fix: check its effect on calibrated strategies


The handoff describes the Pf layout fix as leaving strategies unaffected. This checks that claim before changing the shared feature builder. Production source files and graph caches are not modified. The proposed recipe uses each organism's feature slots, retaining the existing rule against duplicate source columns.

In [1]:
import pandas as pd
from unittest.mock import patch
from starplast import embedding as E, slots, organisms as O
from starplast.strategies import Context
nodes = pd.read_parquet('starplast/data/pf_nodes.parquet')
targets = (None,) + O.get('Pf').targets
before = {target: Context(nodes, organism='Pf').blocks(target) for target in targets}
old_spec = E.default_spec(nodes)
old_columns = set(c for cols in E.columns_for(nodes, old_spec).values() for c in cols)
print('Current display source columns:', len(old_columns))

Current display source columns: 20


In [2]:
def proposed_spec(nodes):
    blocks, seen = [], set()
    for slot in slots.all_slots(slots.table_organism(nodes)):
        if slot.unit != 'gene' or slot.role != 'feature':
            continue
        columns = slots.source_columns(nodes, slot)
        if columns and not set(columns) & seen:
            blocks.append(slot.key)
            seen.update(columns)
    return E.EmbeddingSpec(name='balanced-display', blocks=tuple(blocks))

pf_slots = {s.key: s for s in slots.all_slots('Pf') if s.unit == 'gene' and s.patterns and s.role == 'feature'}
with patch.dict(E.SLOT_BLOCKS, pf_slots), patch.dict(E.BLOCKS, {k: '' for k in pf_slots}), patch.object(E, 'default_spec', proposed_spec):
    matrix, names, rows = E.build_matrix(nodes, E.default_spec(nodes), log=lambda *a: None)
    after = {target: Context(nodes, organism='Pf').blocks(target) for target in targets}
print('Proposed display resolved features:', len(names), 'genes:', len(matrix))

Proposed display resolved features: 118 genes: 5720


In [3]:
records = []
for target in targets:
    old = set(c for cols in before[target].values() for c in cols)
    new = set(c for cols in after[target].values() for c in cols)
    records.append({'target': target or '(none)', 'old_blocks': len(before[target]), 'new_blocks': len(after[target]), 'old_columns': len(old), 'new_columns': len(new), 'added': ', '.join(sorted(new-old)), 'removed': ', '.join(sorted(old-new))})
pd.DataFrame(records)

,target,old_blocks,new_blocks,old_columns,new_columns,added,removed
0,(none),50,62,144,145,"cdpk1_dependent_sites, n_ip_ms_partners, n_lactylsites, n_tcell_epitopes, resistance_target_compounds","idc_cycling_amplitude, idc_peak_hour, n_publications, stage_margin_derived"
1,lopit_pf_location,48,60,140,142,"cdpk1_dependent_sites, hsp90_dependent, n_ip_ms_partners, n_lactylsites, n_tcell_epitopes, resistance_target_compounds","idc_cycling_amplitude, idc_peak_hour, n_publications, stage_margin_derived"
2,pb_transferred_phenotype,48,59,139,140,"cdpk1_dependent_sites, n_ip_ms_partners, n_lactylsites, n_tcell_epitopes, resistance_target_compounds","idc_cycling_amplitude, idc_peak_hour, n_publications, stage_margin_derived"
3,stage_enriched_derived,44,49,120,117,"cdpk1_dependent_sites, n_ip_ms_partners, n_lactylsites, n_tcell_epitopes, resistance_target_compounds","idc_cycling_amplitude, idc_peak_hour, n_publications, riboseq_mrna_early_trophozoite, riboseq_mrna_late_trophozoite, riboseq_mrna_merozoite, riboseq_mrna_ring, riboseq_mrna_schizont"
4,is_exported,48,61,140,143,"cdpk1_dependent_sites, is_palmitoylated, is_tm, n_ip_ms_partners, n_lactylsites, n_tcell_epitopes, resistance_target_compounds","idc_cycling_amplitude, idc_peak_hour, n_publications, stage_margin_derived"


The proposed change alters strategy block grouping and selected columns for every tested holdout. It must therefore be accompanied by a strategy/calibration audit and fresh affected calibration results, or by a deliberate separation of display recipes from the existing calibrated inference recipes. The old calibration must not silently describe these new groupings. This check does not estimate predictive performance.